# NanoQuant · Qwen3.8-27B · 0.55 bpw · Google Colab

Этот ноутбук клонирует [твой форк NanoQuant](https://github.com/rkuznetsovcg/NanoQuant), ставит нужные зависимости, берёт уже подготовленный калибровочный набор из репозитория, запускает профиль `qwen38-0.55-balanced` и сохраняет итоговый `.pt` в Google Drive.

**Как запустить:** в Colab выбери `Среда выполнения → Сменить тип среды выполнения → GPU` и доступный GPU с памятью около 80 ГБ (H100 предпочтителен). Затем выбери `Среда выполнения → Выполнить все` и подтверди доступ к Google Drive. Ячейки проверят, хватает ли памяти и диска, прежде чем скачивать веса модели.

Конфигурация квантования и набор из 128 окон длиной 2 048 токенов берутся из форка. После квантования отдельная ячейка оценивает готовый чекпойнт на WikiText-2 и шести zero-shot задачах; повторной квантизации она не запускает.

Возобновляющие снимки блоков по умолчанию пишутся на локальный диск Colab ради скорости: Drive медленный для этих многогигабайтных временных файлов. Если сама VM Colab будет уничтожена, локальные снимки пропадут. Для сохранения снимков между VM включи `PERSIST_RESUME_ON_DRIVE` в следующей ячейке; это может заметно увеличить время записи.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/rkuznetsovcg/NanoQuant.git"
REPO_BRANCH = "main"
MODEL_ID = "Qwen/Qwen3.8-27B"
PROFILE_RELATIVE_PATH = "configs/qwen38-0.55-balanced.json"
RUN_NAME = "qwen38-27b-0.55-balanced"

# False ускоряет запись промежуточных чекпойнтов. Поставь True до первого запуска,
# если нужно переживать полное уничтожение Colab VM; Drive будет получать крупные снимки блоков.
PERSIST_RESUME_ON_DRIVE = False

# Защита от старта на слишком маленьком runtime. Не снижай пороги, если не уверен в памяти.
MIN_GPU_VRAM_GIB = 70.0
MIN_HOST_RAM_TOTAL_GIB = 70.0
MIN_HOST_RAM_AVAILABLE_GIB = 65.0
MIN_FREE_DISK_GIB = 120.0

ENABLE_FLASHQLA_WHEN_COMPATIBLE = True
DRIVE_MOUNT = Path("/content/drive")
DRIVE_RUN_DIR = DRIVE_MOUNT / "MyDrive" / "NanoQuant" / "runs" / RUN_NAME
REPO_DIR = Path("/content/NanoQuant")
LOCAL_RUN_DIR = Path("/content/nanoquant") / RUN_NAME
LOCAL_OUTPUT = LOCAL_RUN_DIR / "Qwen3.8-27B-NQ-0.55-balanced.pt"


In [ ]:
import os
import re
import shutil
from pathlib import Path

import torch


def gib(value):
    return value / (1024 ** 3)


def read_meminfo_gib(label):
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith(label + ":"):
            return int(line.split()[1]) * 1024 / (1024 ** 3)
    raise RuntimeError(f"Не удалось прочитать {label} из /proc/meminfo")


if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU не найден. В Colab выбери Runtime → Change runtime type → GPU и запусти всё снова.")

props = torch.cuda.get_device_properties(0)
gpu_vram_gib = gib(props.total_memory)
host_total_gib = read_meminfo_gib("MemTotal")
host_available_gib = read_meminfo_gib("MemAvailable")
disk_free_gib = gib(shutil.disk_usage("/content").free)
capability = torch.cuda.get_device_capability(0)
cuda_version = torch.version.cuda or "нет CUDA runtime"

def version_tuple(value):
    return tuple(int(part) for part in re.findall(r"\d+", value.split("+", 1)[0])[:3])

flashqla_compatible = (
    ENABLE_FLASHQLA_WHEN_COMPATIBLE
    and capability >= (9, 0)
    and version_tuple(torch.__version__) >= (2, 8)
    and version_tuple(cuda_version) >= (12, 8)
)

print(f"GPU: {props.name} | {gpu_vram_gib:.1f} GiB | compute {capability[0]}.{capability[1]}")
print(f"PyTorch {torch.__version__} | CUDA runtime {cuda_version}")
print(f"RAM: {host_available_gib:.1f} GiB available / {host_total_gib:.1f} GiB total")
print(f"Free /content disk: {disk_free_gib:.1f} GiB")
print(f"FlashQLA install profile: {'eligible' if flashqla_compatible else 'standard Qwen fast kernels'}")

problems = []
if gpu_vram_gib < MIN_GPU_VRAM_GIB:
    problems.append(f"нужно не менее {MIN_GPU_VRAM_GIB:.0f} GiB GPU VRAM")
if host_total_gib < MIN_HOST_RAM_TOTAL_GIB:
    problems.append(f"нужно не менее {MIN_HOST_RAM_TOTAL_GIB:.0f} GiB общей RAM: BF16-веса сначала загружаются в RAM")
if host_available_gib < MIN_HOST_RAM_AVAILABLE_GIB:
    problems.append(f"нужно не менее {MIN_HOST_RAM_AVAILABLE_GIB:.0f} GiB свободной RAM")
if disk_free_gib < MIN_FREE_DISK_GIB:
    problems.append(f"нужно не менее {MIN_FREE_DISK_GIB:.0f} GiB свободного диска под веса и промежуточные блоки")
if problems:
    raise RuntimeError("Этот runtime мал для выбранного запуска: " + "; ".join(problems))


## Подключение Google Drive

Drive нужен для итогового чекпойнта и файла с настройками запуска. Саму модель и временную работу NanoQuant хранит в диске Colab, чтобы не читать и писать веса через сетевую файловую систему Drive.


In [ ]:
from google.colab import drive

DRIVE_MOUNT.mkdir(parents=True, exist_ok=True)
drive.mount(str(DRIVE_MOUNT))
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Итоговые файлы будут сохранены в: {DRIVE_RUN_DIR}")


## Получение форка и калибровочного набора

Набор уже лежит в GitHub-репозитории, поэтому здесь не нужно отдельно скачивать датасеты или собирать их заново.


In [ ]:
import os
import subprocess
import sys


def run_command(command, cwd=None, env=None):
    print("$", " ".join(map(str, command)))
    command_env = os.environ.copy() if env is None else env.copy()
    source_dir = REPO_DIR / "src"
    if source_dir.is_dir():
        old_pythonpath = command_env.get("PYTHONPATH", "")
        command_env["PYTHONPATH"] = str(source_dir) + (os.pathsep + old_pythonpath if old_pythonpath else "")
    subprocess.run(command, cwd=cwd, env=command_env, check=True)


if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run_command(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    run_command(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])
    run_command(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

PROFILE_PATH = REPO_DIR / PROFILE_RELATIVE_PATH
CALIBRATION_PATH = REPO_DIR / "data/calibration/generated/qwen3.8-27b-multidomain-128x2048/dataset"
if not PROFILE_PATH.is_file():
    raise FileNotFoundError(f"В форке нет профиля: {PROFILE_PATH}")
if not (CALIBRATION_PATH / "dataset_info.json").is_file():
    raise FileNotFoundError(f"В форке нет калибровочного набора: {CALIBRATION_PATH}")

# Keep source imports available even if Colab's editable-install path is not refreshed.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

GIT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(f"NanoQuant commit: {GIT_COMMIT}")
print(f"Профиль: {PROFILE_PATH}")
print(f"Калибровка: {CALIBRATION_PATH}")


## Установка CUDA-зависимостей

Для H100 ноутбук сначала пробует совместимый профиль с FlashQLA для Gated DeltaNet; если установка не проходит, он автоматически ставит стандартные быстрые Qwen-ядра и оставляет линейное внимание на FLA. FlashQLA использует тот же BF16-путь модели и включается только на совместимой связке GPU, CUDA и PyTorch.


In [ ]:
import os
import sys


def install_extra(extra):
    command = [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-cache-dir", "-e", f".[{extra}]"]
    print("$", " ".join(command))
    return subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy()).returncode


if flashqla_compatible:
    status = install_extra("qwen-flashqla")
    if status != 0:
        print("FlashQLA не собрался в этом runtime; включаю резервный путь FLA Triton + kernels.")
        os.environ["FLA_FLASH_QLA"] = "0"
        status = install_extra("qwen-fast,kernels")
else:
    status = install_extra("qwen-fast,kernels")

if status != 0:
    raise RuntimeError("Не удалось установить зависимости NanoQuant. См. вывод pip выше.")
print("Зависимости установлены.")


## Проверка модели, зависимостей и калибровки

Эта ячейка загружает только конфигурацию модели и локальный небольшой калибровочный набор. Полные 27B BF16-веса начнут скачиваться только в следующей ячейке.


In [ ]:
import json
import sys
from datasets import load_from_disk
from transformers import AutoConfig

# The project uses a src/ layout; add it explicitly for this notebook kernel.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

from nanoquant.utils.attention_utils import resolve_attention_backend
from nanoquant.utils.load_utils import _check_fast_linear_attention

profile = json.loads(PROFILE_PATH.read_text())
if profile["model_id"] != MODEL_ID:
    raise ValueError(f"Профиль указывает неожиданную модель: {profile['model_id']}")
if profile["bits"] != 0.55 or profile["num_calib_samples"] != 128 or profile["seqlen"] != 2048:
    raise ValueError("Профиль больше не совпадает с запланированным запуском 0.55 bpw / 128 × 2048")

model_config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
if model_config.model_type not in {"qwen3_5", "qwen3_5_text"}:
    raise RuntimeError(f"Неожиданный тип модели: {model_config.model_type}")
_check_fast_linear_attention(model_config, required=True)
attention_backend = resolve_attention_backend(model_config, profile.get("attn_implementation", "auto"))

calibration = load_from_disk(str(CALIBRATION_PATH))
if len(calibration) < profile["num_calib_samples"]:
    raise ValueError(f"В датасете только {len(calibration)} строк, нужно {profile['num_calib_samples']}")
first_window = calibration[0]["input_ids"]
if len(first_window) != profile["seqlen"]:
    raise ValueError(f"Длина окна {len(first_window)}, ожидалось {profile['seqlen']}")

LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
resume_path = (DRIVE_RUN_DIR / "resume") if PERSIST_RESUME_ON_DRIVE else (LOCAL_RUN_DIR / "resume")
resume_path.mkdir(parents=True, exist_ok=True)

profile["model_id"] = MODEL_ID
profile["calib_dataset"] = str(CALIBRATION_PATH)
profile["qmodel_path"] = str(LOCAL_OUTPUT)
profile["resume_dir"] = str(resume_path)
profile["attn_implementation"] = "auto"
CONFIG_PATH = LOCAL_RUN_DIR / "qwen38-0.55-balanced-colab.json"
CONFIG_PATH.write_text(json.dumps(profile, indent=2) + "\n")

print(f"Модель: {model_config.model_type}; параметры: {model_config.text_config.num_hidden_layers if hasattr(model_config, 'text_config') else model_config.num_hidden_layers} слоёв")
print(f"Полное внимание: {attention_backend}")
print(f"Калибровка: {len(calibration)} окон; по {len(first_window)} токенов")
print(f"Resume: {resume_path}")
print(f"Промежуточная конфигурация: {CONFIG_PATH}")


## Запуск квантования и сохранение результата

Запуск использует тот же профиль из репозитория. Если процесс прервётся, но Colab VM останется живой, снова выполни последнюю ячейку: NanoQuant продолжит с сохранённого блока. Под ячейкой будет обновляться панель текущего этапа, прогресс-блоков, занятость GPU и последние сообщения. Полный лог сохранится в Drive; при сбое ячейка покажет настоящую причину из процесса. После успешного завершения ноутбук скопирует чекпойнт и метаданные в Drive.

Если появится ошибка о нехватке системной RAM, нужен runtime с большим объёмом **обычной RAM**, даже если у GPU 80 ГБ VRAM. Веса этой модели в BF16 сначала загружаются в системную память.


In [ ]:
import codecs
import datetime
import errno
import fcntl
import html
import json
import os
import pty
import re
import selectors
import shutil
import struct
import subprocess
import sys
import termios
import time
from collections import deque
from IPython.display import FileLink, HTML, display

DRIVE_OUTPUT = DRIVE_RUN_DIR / LOCAL_OUTPUT.name
DRIVE_COMPLETE = DRIVE_RUN_DIR / "run_complete.json"
DRIVE_TEMP_OUTPUT = DRIVE_RUN_DIR / (LOCAL_OUTPUT.name + ".uploading")
LOCAL_LOG = LOCAL_RUN_DIR / "nanoquant.log"
DRIVE_LOG = DRIVE_RUN_DIR / "nanoquant.log"
ANSI_RE = re.compile(r"\x1b(?:\[[0-?]*[ -/]*[@-~]|\][^\x07]*(?:\x07|\x1b\\))")


def run_nanoquant_live(command, cwd, log_path):
    state = {"stage": "Запуск NanoQuant", "progress": None, "line": "Подключаю вывод процесса…"}
    recent = deque(maxlen=8)
    started = time.monotonic()

    def consume_line(raw_line):
        line = ANSI_RE.sub("", raw_line).strip()
        if not line:
            return
        state["line"] = line
        recent.append(line)
        lower = line.lower()
        if "loading model" in lower or "downloading" in lower:
            state["stage"] = "Загрузка весов модели"
            state["progress"] = None
        elif "calibration strategy" in lower or "single pass:" in lower:
            state["stage"] = "Сбор статистики калибровки"
            state["progress"] = None
        elif "caching initial inputs" in lower:
            state["stage"] = "Кеширование входов блоков"
            state["progress"] = None
        elif "compressing layers" in lower:
            state["stage"] = "Квантование и реконструкция блоков"
        elif "tuning non-factorized" in lower:
            state["stage"] = "Настройка нефакторизованных весов"
        elif "initialization via admm" in lower:
            state["stage"] = "Факторизация весов (ADMM)"
        elif "tuning factorized" in lower:
            state["stage"] = "Настройка факторизованных весов"
        elif "caching teacher hidden states" in lower or "model-level kd" in lower:
            state["stage"] = "Финальная дистилляция модели"
        elif "saved quantized model" in lower:
            state["stage"] = "Сохранение чекпойнта"

        match = re.search(r"(\d{1,3})%\|.*?\|\s*(\d+)\s*/\s*(\d+)", line)
        if match:
            state["progress"] = (int(match.group(2)), int(match.group(3)))
        elif state["stage"] == "Сбор статистики калибровки":
            match = re.search(r"\|\s*(\d+)\s*/\s*(\d+)", line)
            if match:
                state["progress"] = (int(match.group(1)), int(match.group(2)))

    def render_panel(status="Работает"):
        elapsed = int(time.monotonic() - started)
        hours, remainder = divmod(elapsed, 3600)
        minutes, seconds = divmod(remainder, 60)
        progress_html = ""
        progress = state["progress"]
        if progress and progress[1] > 0:
            current, total = progress
            percent = 100 * current / total
            progress_html = (
                f'<progress value="{current}" max="{total}" '
                'style="width:100%;height:18px"></progress>'
                f'<div>{current} / {total} · {percent:.1f}%</div>'
            )
        try:
            free_vram, total_vram = torch.cuda.mem_get_info(0)
            gpu_text = f"{(total_vram-free_vram)/(1024**3):.1f} / {total_vram/(1024**3):.1f} GiB"
        except Exception:
            gpu_text = "считывание недоступно"
        try:
            free_disk = shutil.disk_usage("/content").free / (1024**3)
            disk_text = f"свободно {free_disk:.1f} GiB"
        except Exception:
            disk_text = "считывание недоступно"
        tail = html.escape("\n".join(recent)[-5000:])
        current_line = html.escape(state["line"][-500:])
        return HTML(
            '<div style="border:1px solid #9aa4b2;border-radius:10px;padding:14px;'
            'font:14px system-ui;max-width:1000px">'
            f'<div style="font-size:18px;font-weight:700">NanoQuant · {html.escape(status)}</div>'
            f'<div style="margin:8px 0"><b>Этап:</b> {html.escape(state["stage"])}</div>'
            f'{progress_html}'
            f'<div style="margin:8px 0;color:#52606d">Время: {hours:02d}:{minutes:02d}:{seconds:02d} '
            f'· GPU: {gpu_text} · Диск: {disk_text}</div>'
            f'<div style="margin:8px 0;overflow-wrap:anywhere">{current_line}</div>'
            f'<details><summary>Последние сообщения процесса</summary>'
            f'<pre style="white-space:pre-wrap;max-height:260px;overflow:auto">{tail}</pre></details>'
            '</div>'
        )

    env = os.environ.copy()
    source_dir = str(Path(cwd) / "src")
    previous_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = source_dir + (os.pathsep + previous_pythonpath if previous_pythonpath else "")
    env["PYTHONUNBUFFERED"] = "1"
    env["TQDM_MININTERVAL"] = "1"
    env["TQDM_MINITERS"] = "1"
    env.setdefault("TERM", "xterm-256color")

    master_fd, slave_fd = pty.openpty()
    fcntl.ioctl(slave_fd, termios.TIOCSWINSZ, struct.pack("HHHH", 32, 140, 0, 0))
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdin=subprocess.DEVNULL,
        stdout=slave_fd, stderr=subprocess.STDOUT, close_fds=True,
    )
    os.close(slave_fd)
    os.set_blocking(master_fd, False)
    selector = selectors.DefaultSelector()
    selector.register(master_fd, selectors.EVENT_READ)
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    pending = ""
    last_update = 0.0
    display_handle = display(render_panel(), display_id=True)

    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("wb") as log_file:
        while process.poll() is None or selector.get_map():
            events = selector.select(timeout=0.5) if selector.get_map() else []
            for key, _ in events:
                try:
                    chunk = os.read(key.fd, 65536)
                except BlockingIOError:
                    continue
                except OSError as error:
                    if error.errno in (errno.EIO, errno.EBADF):
                        selector.unregister(key.fd)
                        continue
                    raise
                if not chunk:
                    selector.unregister(key.fd)
                    continue
                log_file.write(chunk)
                log_file.flush()
                pending += decoder.decode(chunk)
                while True:
                    separators = [position for position in (pending.find("\r"), pending.find("\n")) if position >= 0]
                    if not separators:
                        break
                    position = min(separators)
                    consume_line(pending[:position])
                    pending = pending[position + 1:]
            now = time.monotonic()
            if not selector.get_map() and process.poll() is None:
                time.sleep(0.2)
            if now - last_update >= 0.8:
                display_handle.update(render_panel())
                last_update = now

        pending += decoder.decode(b"", final=True)
        if pending.strip():
            consume_line(pending)

    selector.close()
    os.close(master_fd)
    return_code = process.wait()
    display_handle.update(render_panel("Завершено" if return_code == 0 else f"Ошибка · код {return_code}"))
    return return_code


if DRIVE_COMPLETE.is_file() and DRIVE_OUTPUT.is_file() and DRIVE_OUTPUT.stat().st_size > 0:
    print(f"Готовый запуск уже сохранён: {DRIVE_OUTPUT}")
    print(f"Метка завершения: {DRIVE_COMPLETE}")
else:
    if LOCAL_OUTPUT.exists():
        LOCAL_OUTPUT.unlink()

    command = [sys.executable, "-m", "nanoquant.main", str(CONFIG_PATH)]
    exit_code = run_nanoquant_live(command, REPO_DIR, LOCAL_LOG)
    try:
        shutil.copy2(LOCAL_LOG, DRIVE_LOG)
    except Exception as copy_error:
        print(f"Не удалось скопировать лог в Drive: {copy_error}")

    if exit_code != 0:
        raw_tail = LOCAL_LOG.read_bytes()[-24000:].decode("utf-8", errors="replace")
        raw_tail = ANSI_RE.sub("", raw_tail).replace("\r", "\n")
        tail_lines = [line for line in raw_tail.splitlines() if line.strip()][-35:]
        display(FileLink(str(LOCAL_LOG), result_html_prefix="Полный лог: "))
        raise RuntimeError(
            f"NanoQuant завершился с кодом {exit_code}. Последние строки настоящей ошибки ниже; "
            f"полный лог: {LOCAL_LOG}\n\n" + "\n".join(tail_lines)
        )

    if not LOCAL_OUTPUT.is_file() or LOCAL_OUTPUT.stat().st_size == 0:
        display(FileLink(str(LOCAL_LOG), result_html_prefix="Полный лог: "))
        raise RuntimeError(f"Команда завершилась без итогового чекпойнта. Лог: {LOCAL_LOG}")

    shutil.copy2(LOCAL_OUTPUT, DRIVE_TEMP_OUTPUT)
    if DRIVE_TEMP_OUTPUT.stat().st_size != LOCAL_OUTPUT.stat().st_size:
        DRIVE_TEMP_OUTPUT.unlink(missing_ok=True)
        raise IOError("Размер файла после копирования в Drive не совпал с исходным")
    DRIVE_TEMP_OUTPUT.replace(DRIVE_OUTPUT)

    run_metadata = {
        "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "model_id": MODEL_ID,
        "profile": PROFILE_RELATIVE_PATH,
        "bits": 0.55,
        "calibration_windows": 128,
        "sequence_length": 2048,
        "attention_backend": attention_backend,
        "gpu": props.name,
        "gpu_vram_gib": round(gpu_vram_gib, 2),
        "pytorch": torch.__version__,
        "cuda_runtime": cuda_version,
        "nanoquant_commit": GIT_COMMIT,
        "checkpoint_file": DRIVE_OUTPUT.name,
        "checkpoint_size_bytes": DRIVE_OUTPUT.stat().st_size,
    }
    temp_metadata = DRIVE_RUN_DIR / "run_complete.json.tmp"
    temp_metadata.write_text(json.dumps(run_metadata, indent=2) + "\n")
    temp_metadata.replace(DRIVE_COMPLETE)

    print("Квантование завершено.")
    print(f"Чекпойнт: {DRIVE_OUTPUT}")
    print(f"Метаданные: {DRIVE_COMPLETE}")
    display(FileLink(str(DRIVE_LOG), result_html_prefix="Лог запуска: "))


### Что будет в Google Drive

В папке `MyDrive/NanoQuant/runs/qwen38-27b-0.55-balanced/` появятся:

- `Qwen3.8-27B-NQ-0.55-balanced.pt` — итоговый чекпойнт;
- `run_complete.json` — профиль, ревизия кода, GPU и параметры запуска;
- `nanoquant.log` — полный лог квантования;
- `evaluation.log` — лог PPL и zero-shot оценки;
- `qwen38-0.55-balanced-eval.json` — конфигурация оценки.

Открыть этот ноутбук позднее можно из форка: [NanoQuant Colab notebook](https://colab.research.google.com/github/rkuznetsovcg/NanoQuant/blob/main/notebooks/qwen38_0.55_colab.ipynb).


## Оценка готового чекпойнта

Эта ячейка загружает уже сохранённый `.pt` и запускает WikiText-2 PPL плюс шесть zero-shot задач из NanoQuant paper. Она проверяет размер чекпойнта по `run_complete.json` до запуска. Результаты и конфигурация оценки сохраняются в папку запуска на Google Drive.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# Эти пути соответствуют RUN_NAME и LOCAL_OUTPUT в начале ноутбука.
EVAL_REPO = Path("/content/NanoQuant")
EVAL_RUN_DIR = Path("/content/drive/MyDrive/NanoQuant/runs/qwen38-27b-0.55-balanced")
EVAL_CHECKPOINT = EVAL_RUN_DIR / "Qwen3.8-27B-NQ-0.55-balanced.pt"
EVAL_METADATA = EVAL_RUN_DIR / "run_complete.json"

if not EVAL_REPO.is_dir() or not (EVAL_REPO / "src/nanoquant/main.py").is_file():
    raise FileNotFoundError("Репозиторий не найден. Сначала запусти ячейки получения форка и установки зависимостей.")
if not EVAL_CHECKPOINT.is_file() or not EVAL_METADATA.is_file():
    raise FileNotFoundError("В Google Drive не найдены чекпойнт и run_complete.json. Сначала заверши квантование.")

metadata = json.loads(EVAL_METADATA.read_text())
expected_size = int(metadata.get("checkpoint_size_bytes", 0))
actual_size = EVAL_CHECKPOINT.stat().st_size
if metadata.get("model_id") != "Qwen/Qwen3.8-27B" or (expected_size and actual_size != expected_size):
    raise ValueError(f"Чекпойнт не совпадает с метаданными: модель={metadata.get('model_id')}, размер={actual_size}, ожидался={expected_size}")

eval_profile = json.loads((EVAL_REPO / "configs/qwen38-0.55-balanced.json").read_text())
eval_profile.update({
    "qmodel_path": str(EVAL_CHECKPOINT),
    "attn_implementation": metadata.get("attention_backend", "auto"),
    "ppl_task": "wikitext2",
    "zeroshot_task": "boolq,piqa,hellaswag,winogrande,arc_easy,arc_challenge",
    "num_fewshot": 0,
    "limit": -1,
    "batch_size": "auto",
    "resume_dir": "",
})
EVAL_CONFIG = EVAL_RUN_DIR / "qwen38-0.55-balanced-eval.json"
EVAL_LOG = EVAL_RUN_DIR / "evaluation.log"
EVAL_CONFIG.write_text(json.dumps(eval_profile, indent=2) + "\n")

eval_env = os.environ.copy()
eval_source = str(EVAL_REPO / "src")
old_pythonpath = eval_env.get("PYTHONPATH", "")
eval_env["PYTHONPATH"] = eval_source + (os.pathsep + old_pythonpath if old_pythonpath else "")
eval_env["PYTHONUNBUFFERED"] = "1"
eval_command = [sys.executable, "-u", "-m", "nanoquant.main", str(EVAL_CONFIG)]
print("Загружаю готовый чекпойнт и запускаю WikiText-2 PPL и zero-shot оценку…")
print(f"Чекпойнт: {EVAL_CHECKPOINT} ({actual_size / (1024 ** 3):.2f} GiB)")
print(f"Полный лог оценки: {EVAL_LOG}")

process = subprocess.Popen(
    eval_command,
    cwd=EVAL_REPO,
    env=eval_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=8192,
)
with EVAL_LOG.open("wb") as log_file:
    while True:
        chunk = process.stdout.read1(8192)
        if not chunk:
            break
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()
        log_file.write(chunk)
        log_file.flush()
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Оценка завершилась с кодом {return_code}. Полный лог: {EVAL_LOG}")
print(f"\nОценка завершена. Лог сохранён: {EVAL_LOG}")
